# 🏗️ Notebook 1: S3 (Object Storage) — Requirements & Architecture

## 🛠️ Setup

```bash
cd 06-system-designs/s3
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## What we're designing

A highly durable object store. Clients create buckets, upload objects (blobs + metadata), download them, list, version, and delete. Objects are immutable blobs addressed by `(bucket, key)` (+ optional version id).

Hard problems: **durability** (11 nines), **scalability** (exabytes), **cost**.

## Requirements

### Functional
- CRUD on buckets and objects.
- Versioning per bucket.
- Replication across regions.
- **Presigned URLs** to grant temporary access without exposing creds.

### Non-functional
- Durability 99.999999999% — lose 1 in 10¹¹ objects/year.
- Eventual consistency tolerated; strong read-after-write on the same key.
- Massive scale, cheap storage.

## Back-of-envelope

- 100 B objects and 1 TB objects both happen → two-tier path (metadata vs data).
- 1 M PUT/s peak. Durability via **erasure coding** (e.g., 10+4) cheaper than 3× replication.

## High-level architecture

```
  [Client]──SigV4/HMAC──►  API Front End (REST)
                               │
          ┌────────────────────┼──────────────────┐
          ▼                    ▼                  ▼
   Metadata Service      Data Plane         Auth/IAM
   (key→shard map)         │
          │                ▼
          │         Storage Nodes (EC shards)
          │         ┌──────┬──────┬──────┬──────┐
          │         │ D1   │ D2   │ D3   │ P1   │ …
          │         └──────┴──────┴──────┴──────┘
          ▼
   Replication (cross-region, async)
```

- **Metadata plane** (smaller, consistent DB) maps keys to data locations.
- **Data plane** stores blobs as erasure-coded shards across racks/AZs.

## Why these choices?

- Each service in the diagram owns one responsibility — easier to scale and reason about.
- Stateless services scale horizontally; stateful stores are chosen per access pattern.
- The next two notebooks zoom into the **data model + APIs** and one **deep-dive algorithm**.